# Implement DDIM Sampling + Classifier-Free Guidance - SOLUTION

**Difficulty**: 🔴 Hard

**Companies**: Midjourney, Runway, Stability AI, Adobe

---

### Problem Statement

DDPM sampling requires iterating through all T timesteps, which is slow. **DDIM** (Denoising Diffusion Implicit Models) provides a deterministic sampling process that can skip timesteps, generating high-quality samples in far fewer steps.

**Classifier-Free Guidance (CFG)** improves conditional generation by training a single model to handle both conditional and unconditional denoising. At inference time, the unconditional prediction is "pushed away from" while the conditional prediction is amplified, controlled by a guidance scale.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Generate training data: 4 class-conditional 2D clusters
torch.manual_seed(42)
np.random.seed(42)

n_per_class = 500
num_classes = 4

centers = torch.tensor([[2.0, 2.0], [-2.0, 2.0], [-2.0, -2.0], [2.0, -2.0]])
all_data = []
all_labels = []
for c in range(num_classes):
    cluster = centers[c] + 0.3 * torch.randn(n_per_class, 2)
    all_data.append(cluster)
    all_labels.append(torch.full((n_per_class,), c, dtype=torch.long))

data = torch.cat(all_data, dim=0)
labels = torch.cat(all_labels, dim=0)
n_samples = data.shape[0]

plt.figure(figsize=(6, 6))
for c in range(num_classes):
    mask = labels == c
    plt.scatter(data[mask, 0].numpy(), data[mask, 1].numpy(), s=5, alpha=0.5, label=f'Class {c}')
plt.legend()
plt.title('Training Data: 4 Class-Conditional Clusters')
plt.axis('equal')
plt.show()

print(f'Data shape: {data.shape}, Labels shape: {labels.shape}')

In [ ]:
# --- Shared: Beta Schedule and Forward Process ---

def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)


def extract(a, t, x_shape):
    batch_size = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def q_sample(x_0, t, noise, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod):
    sqrt_alpha_bar = extract(sqrt_alphas_cumprod, t, x_0.shape)
    sqrt_one_minus_alpha_bar = extract(sqrt_one_minus_alphas_cumprod, t, x_0.shape)
    return sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise


T = 300
betas = linear_beta_schedule(T)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

print('Schedule precomputed.')

In [ ]:
# --- Part 1: Class-Conditional Denoiser with Label Dropout ---

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))


class ConditionalDenoiser(nn.Module):
    """
    Class-conditional denoiser that predicts noise given (x_t, t, class_label).
    Uses label embedding with a special "null" class (index = num_classes) for CFG.
    """
    def __init__(self, data_dim=2, num_classes=4, hidden_dim=128, time_dim=64, label_dropout=0.15):
        super().__init__()
        self.label_dropout = label_dropout
        self.num_classes = num_classes

        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.GELU(),
        )

        # Class embedding: num_classes + 1 for the null/unconditional class
        self.class_emb = nn.Embedding(num_classes + 1, time_dim)

        # Projections
        self.input_proj = nn.Linear(data_dim, hidden_dim)
        self.time_proj = nn.Linear(time_dim, hidden_dim)
        self.class_proj = nn.Linear(time_dim, hidden_dim)

        self.net = nn.Sequential(
            ResidualBlock(hidden_dim),
            ResidualBlock(hidden_dim),
            ResidualBlock(hidden_dim),
        )

        self.output_proj = nn.Linear(hidden_dim, data_dim)

    def forward(self, x, t, class_label):
        """
        Args:
            x (Tensor): Noised input, shape (B, data_dim).
            t (Tensor): Timesteps, shape (B,).
            class_label (Tensor): Class labels, shape (B,). Use num_classes for unconditional.

        Returns:
            Tensor: Predicted noise, shape (B, data_dim).
        """
        # During training, randomly replace labels with null class for CFG
        if self.training:
            mask = torch.rand(class_label.shape[0]) < self.label_dropout
            class_label = class_label.clone()
            class_label[mask] = self.num_classes  # null class

        # Embeddings
        t_emb = self.time_mlp(t)
        c_emb = self.class_emb(class_label)

        # Combine input, time, and class
        h = self.input_proj(x) + self.time_proj(t_emb) + self.class_proj(c_emb)
        h = self.net(h)
        return self.output_proj(h)


model = ConditionalDenoiser(data_dim=2, num_classes=num_classes, hidden_dim=128, time_dim=64)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# --- Part 2: Training ---

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 400
batch_size = 256
losses = []

model.train()
for epoch in range(n_epochs):
    perm = torch.randperm(n_samples)
    epoch_loss = 0.0
    n_batches = 0

    for i in range(0, n_samples, batch_size):
        x_0 = data[perm[i:i+batch_size]]
        y = labels[perm[i:i+batch_size]]
        b = x_0.shape[0]

        t = torch.randint(0, T, (b,))
        noise = torch.randn_like(x_0)
        x_t = q_sample(x_0, t, noise, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod)

        predicted_noise = model(x_t, t, y)
        loss = F.mse_loss(predicted_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.6f}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
# --- Part 3: Classifier-Free Guidance Predict ---

def guided_predict(model, x_t, t_batch, class_label, guidance_scale, num_classes=4):
    """
    Combine conditional and unconditional predictions for classifier-free guidance.

    Args:
        model: Trained conditional denoiser.
        x_t (Tensor): Current sample, shape (B, D).
        t_batch (Tensor): Timestep batch, shape (B,).
        class_label (Tensor): Class labels, shape (B,).
        guidance_scale (float): How much to amplify conditional signal.
        num_classes (int): Number of real classes.

    Returns:
        eps_guided (Tensor): Guided noise prediction.
    """
    # Conditional prediction
    eps_cond = model(x_t, t_batch, class_label)

    # Unconditional prediction (null class = num_classes)
    null_labels = torch.full_like(class_label, num_classes)
    eps_uncond = model(x_t, t_batch, null_labels)

    # Guided combination
    eps_guided = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
    return eps_guided

In [ ]:
# --- Part 4: DDIM Sampling Step ---

def ddim_sample_step(model, x_t, t, t_prev, alphas_cumprod, class_label, guidance_scale=1.0, num_classes=4):
    """
    One step of DDIM (deterministic, eta=0) sampling with optional classifier-free guidance.

    Args:
        model: Trained conditional denoiser.
        x_t (Tensor): Current sample, shape (B, D).
        t (int): Current timestep.
        t_prev (int): Previous timestep (t_prev < t). Use -1 for final step.
        alphas_cumprod (Tensor): Cumulative product of alphas.
        class_label (Tensor): Class labels for conditional generation, shape (B,).
        guidance_scale (float): CFG scale. 1.0 = no guidance.
        num_classes (int): Number of real classes.

    Returns:
        x_{t_prev} (Tensor): Denoised sample.
    """
    b = x_t.shape[0]
    t_batch = torch.full((b,), t, dtype=torch.long)

    # Get noise prediction (with or without CFG)
    if guidance_scale != 1.0:
        eps = guided_predict(model, x_t, t_batch, class_label, guidance_scale, num_classes)
    else:
        eps = model(x_t, t_batch, class_label)

    # Get alpha_bar values
    alpha_bar_t = alphas_cumprod[t]

    # Predict x_0 from x_t and predicted noise
    x0_pred = (x_t - torch.sqrt(1.0 - alpha_bar_t) * eps) / torch.sqrt(alpha_bar_t)

    # Compute x_{t_prev} using DDIM formula (eta=0, deterministic)
    if t_prev >= 0:
        alpha_bar_t_prev = alphas_cumprod[t_prev]
        x_t_prev = (
            torch.sqrt(alpha_bar_t_prev) * x0_pred
            + torch.sqrt(1.0 - alpha_bar_t_prev) * eps
        )
    else:
        # Final step: return predicted x_0
        x_t_prev = x0_pred

    return x_t_prev

In [ ]:
# --- Part 5: Full DDIM Sampling Loop ---

@torch.no_grad()
def ddim_sample_loop(model, shape, alphas_cumprod, class_label, num_steps=50,
                     guidance_scale=1.0, num_classes=4, total_timesteps=300):
    """
    Full DDIM sampling loop with optional CFG.

    Args:
        model: Trained denoiser.
        shape (tuple): Shape of output, e.g. (n_samples, 2).
        alphas_cumprod (Tensor): Precomputed schedule.
        class_label (Tensor): Class labels for each sample.
        num_steps (int): Number of DDIM steps (can be << T).
        guidance_scale (float): CFG guidance scale.
        num_classes (int): Number of real classes.
        total_timesteps (int): Total training timesteps T.

    Returns:
        samples (Tensor): Generated samples.
    """
    # Create sub-sequence of timesteps (evenly spaced)
    step_size = total_timesteps // num_steps
    timestep_seq = list(range(0, total_timesteps, step_size))

    # Start from pure noise
    x = torch.randn(shape)

    # Iterate through timestep pairs in reverse
    for i in reversed(range(len(timestep_seq))):
        t = timestep_seq[i]
        t_prev = timestep_seq[i - 1] if i > 0 else -1
        x = ddim_sample_step(model, x, t, t_prev, alphas_cumprod, class_label,
                            guidance_scale, num_classes)

    return x


print('DDIM sample loop defined.')

In [ ]:
# --- Validation ---

model.eval()

# Generate samples for each class with different guidance scales
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
guidance_scales = [1.0, 3.0, 7.0]
ddim_steps_list = [50, 50, 50]

n_gen = 300

for col, (gs, steps) in enumerate(zip(guidance_scales, ddim_steps_list)):
    all_samples = []
    all_labels_gen = []
    for c in range(num_classes):
        class_label = torch.full((n_gen // num_classes,), c, dtype=torch.long)
        samples = ddim_sample_loop(
            model, (n_gen // num_classes, 2), alphas_cumprod, class_label,
            num_steps=steps, guidance_scale=gs, num_classes=num_classes, total_timesteps=T
        )
        all_samples.append(samples)
        all_labels_gen.append(class_label)

    all_samples = torch.cat(all_samples, dim=0)
    all_labels_gen = torch.cat(all_labels_gen, dim=0)

    for c in range(num_classes):
        mask = all_labels_gen == c
        axes[0, col].scatter(all_samples[mask, 0].numpy(), all_samples[mask, 1].numpy(),
                           s=10, alpha=0.5, label=f'Class {c}')
    axes[0, col].set_title(f'Generated (guidance={gs}, steps={steps})')
    axes[0, col].set_xlim(-4, 4)
    axes[0, col].set_ylim(-4, 4)
    axes[0, col].legend()
    axes[0, col].set_aspect('equal')

# Plot training data for reference
for c in range(num_classes):
    mask = labels == c
    axes[1, 0].scatter(data[mask, 0].numpy(), data[mask, 1].numpy(), s=5, alpha=0.5, label=f'Class {c}')
axes[1, 0].set_title('Training Data')
axes[1, 0].set_xlim(-4, 4)
axes[1, 0].set_ylim(-4, 4)
axes[1, 0].legend()
axes[1, 0].set_aspect('equal')

# Compare DDIM steps: 10 vs 300
for col_offset, steps in enumerate([10, 300]):
    all_samples = []
    all_labels_gen = []
    for c in range(num_classes):
        class_label = torch.full((n_gen // num_classes,), c, dtype=torch.long)
        samples = ddim_sample_loop(
            model, (n_gen // num_classes, 2), alphas_cumprod, class_label,
            num_steps=steps, guidance_scale=3.0, num_classes=num_classes, total_timesteps=T
        )
        all_samples.append(samples)
        all_labels_gen.append(class_label)

    all_samples = torch.cat(all_samples, dim=0)
    all_labels_gen = torch.cat(all_labels_gen, dim=0)

    for c in range(num_classes):
        mask = all_labels_gen == c
        axes[1, col_offset + 1].scatter(all_samples[mask, 0].numpy(), all_samples[mask, 1].numpy(),
                                        s=10, alpha=0.5, label=f'Class {c}')
    axes[1, col_offset + 1].set_title(f'DDIM steps={steps}, guidance=3.0')
    axes[1, col_offset + 1].set_xlim(-4, 4)
    axes[1, col_offset + 1].set_ylim(-4, 4)
    axes[1, col_offset + 1].legend()
    axes[1, col_offset + 1].set_aspect('equal')

plt.tight_layout()
plt.show()

# Checks
print('=== Validation ===')
assert losses[-1] < losses[0], 'Training loss should decrease!'
print('PASSED: Loss decreased during training')

test_label = torch.full((100,), 0, dtype=torch.long)
test_samples = ddim_sample_loop(
    model, (100, 2), alphas_cumprod, test_label,
    num_steps=50, guidance_scale=3.0, num_classes=num_classes, total_timesteps=T
)
assert test_samples.abs().max() < 15, 'Generated samples have extreme values!'
print('PASSED: Generated samples in reasonable range')

print('\nAll validations passed!')